In [0]:
dbutils.widgets.text("catalog", "sunny_bay_roastery")
catalog = dbutils.widgets.get("catalog")

dbutils.widgets.text("gold_schema", "gold")
gold_schema = dbutils.widgets.get("gold_schema")

dbutils.widgets.text("prefix", "")
prefix = dbutils.widgets.get("prefix")

In [0]:
# Pre-document the gold tables so workshop participants land in a well-governed
# catalog: every gold table (except dim_customer) gets a rich table description,
# per-column comments, and governance tags.
#
# dim_customer is deliberately LEFT UNDOCUMENTED — it is the hands-on exercise in
# Lab 1, where participants add its description and a tag with AI assistance.
#
# Each statement runs independently and tolerates errors, so this step can never
# fail the setup job even if a metadata operation is unsupported on a workspace.

METADATA = {
    "fact_coffee_sales": {
        "comment": "**Coffee sales fact table (gold).** One row per sold line item at Sunny Bay Roastery, enriched with gross/net revenue, VAT and cost of goods. The analytical heart of the star schema — dashboards, metric views and Genie answers all build on this table.",
        "tags": {"domain": "sales", "layer": "gold", "certified": "true"},
        "columns": {
            "date_key": "Foreign key to dim_date (the date of the sale).",
            "store_key": "Foreign key to dim_store (where the sale happened).",
            "product_key": "Foreign key to dim_product (what was sold).",
            "customer_key": "Foreign key to dim_customer (who bought).",
            "quantity_sold": "Number of units sold on this line item.",
            "gross_revenue_usd": "Gross revenue in USD (list price x quantity), before tax.",
            "net_revenue_usd": "Net revenue in USD after removing VAT.",
            "vat_usd": "Value-added tax charged on this line item, in USD.",
            "cost_of_goods_usd": "Cost of goods sold for this line item, in USD.",
        },
    },
    "dim_product": {
        "comment": "**Product dimension (gold).** The coffee, beans and gear Sunny Bay Roastery sells — one row per product, with category, availability, list price and unit cost.",
        "tags": {"domain": "sales", "layer": "gold", "certified": "true"},
        "columns": {
            "product_key": "Surrogate key for the product.",
            "product_name": "Product display name.",
            "product_category": "High-level product category (e.g. Beans, Drinks, Gear).",
            "product_subcategory": "More specific grouping within the category.",
            "is_beans": "True if the product is coffee beans.",
            "available_in_store": "True if sold in the physical cafes.",
            "available_online": "True if sold through the online channel.",
            "list_price_usd": "Retail list price per unit, in USD.",
            "cost_of_goods_usd": "Unit cost of goods, in USD.",
        },
    },
    "dim_store": {
        "comment": "**Store dimension (gold).** Sunny Bay Roastery's cafes and its online channel — one row per store, with location, size and the tax rate applied to its sales.",
        "tags": {"domain": "sales", "layer": "gold", "certified": "true"},
        "columns": {
            "store_key": "Surrogate key for the store.",
            "store_name": "Store display name.",
            "store_type": "Store format (e.g. cafe, kiosk, online).",
            "city": "City where the store is located.",
            "neighborhood_or_channel": "Neighborhood (physical) or channel name (online).",
            "is_online": "True for the online channel, false for physical cafes.",
            "store_area_sqm": "Floor area in square metres.",
            "seating_capacity": "Number of seats.",
            "num_employees": "Headcount at the store.",
            "store_manager": "Name of the store manager.",
            "tax_rate": "VAT rate applied to this store's sales.",
            "country_name": "Country name.",
            "country_iso2": "ISO 3166-1 alpha-2 country code.",
            "country_iso3": "ISO 3166-1 alpha-3 country code.",
            "state_province": "State or province.",
            "state_iso2": "ISO state/province code.",
            "county_district": "County or district.",
            "postal_code": "Postal code.",
            "latitude": "Latitude of the store location.",
            "longitude": "Longitude of the store location.",
        },
    },
    "dim_date": {
        "comment": "**Date dimension (gold).** One row per calendar day, with the date parts, week, season, weekend and US public-holiday flags used to slice sales over time.",
        "tags": {"domain": "sales", "layer": "gold", "certified": "true"},
        "columns": {
            "date_key": "Surrogate key for the date (YYYYMMDD).",
            "date": "Calendar date.",
            "year": "Calendar year.",
            "month": "Month number (1-12).",
            "day": "Day of the month.",
            "calendar_week": "ISO calendar week number.",
            "day_of_week": "Day-of-week number.",
            "day_name": "Day name (e.g. Monday).",
            "is_weekend": "True if the day is a weekend.",
            "season": "Meteorological season.",
            "is_us_public_holiday": "True if the day is a US public holiday.",
        },
    },
}


def s(text):
    """Quote a string as a SQL literal."""
    return "'" + text.replace("'", "''") + "'"


def fq(name):
    return f"`{catalog}`.`{gold_schema}`.`{prefix}{name}`"


def run(sql, label):
    try:
        spark.sql(sql)
        return True
    except Exception as e:
        print(f"  ⚠️  skipped {label}: {str(e).splitlines()[0][:160]}")
        return False


for table, meta in METADATA.items():
    t = fq(table)
    run(f"COMMENT ON TABLE {t} IS {s(meta['comment'])}", f"{table} table comment")
    for col, desc in meta["columns"].items():
        run(f"ALTER TABLE {t} ALTER COLUMN `{col}` COMMENT {s(desc)}", f"{table}.{col}")
    if meta.get("tags"):
        tag_sql = ", ".join(f"{s(k)} = {s(v)}" for k, v in meta["tags"].items())
        run(f"ALTER TABLE {t} SET TAGS ({tag_sql})", f"{table} tags")
    print(f"✅ Documented {table}")

print()
print("✅ Gold metadata applied. dim_customer was left undocumented on purpose "
      "(Lab 1 hands-on exercise).")
